# Recorte da cidade-alvo e exportação inicial

Este notebook prepara o primeiro recorte do POC a partir do snapshot mensal do CNPJ.

## Objetivos

- definir a cidade-alvo
- ler empresas, estabelecimentos e simples
- filtrar o município de interesse
- unir domínios principais
- exportar artefatos bronze/silver para o POC

In [ ]:
from pathlib import Path

try:
    import polars as pl
except ImportError:
    !pip -q install polars pyarrow
    import polars as pl

RAIZ_DADOS = Path('/content/drive/MyDrive/geoBusiness/dados_brutos/cnpj')
SNAPSHOT_MES = '2026-04'
CODIGO_MUNICIPIO = '7107'
SNAPSHOT = RAIZ_DADOS / SNAPSHOT_MES
SNAPSHOT

In [ ]:
colunas_empresas = [
    'cnpj_basico', 'razao_social', 'natureza_juridica', 'qualificacao_responsavel',
    'capital_social', 'porte_empresa', 'ente_federativo_responsavel'
]
colunas_estabelecimentos = [
    'cnpj_basico', 'cnpj_ordem', 'cnpj_dv', 'identificador_matriz_filial', 'nome_fantasia',
    'situacao_cadastral', 'data_situacao_cadastral', 'motivo_situacao_cadastral',
    'nome_cidade_exterior', 'pais', 'data_inicio_atividade', 'cnae_fiscal_principal',
    'cnaes_secundarios', 'tipo_logradouro', 'logradouro', 'numero', 'complemento', 'bairro',
    'cep', 'uf', 'municipio', 'ddd_1', 'telefone_1', 'ddd_2', 'telefone_2', 'ddd_fax', 'fax',
    'correio_eletronico', 'situacao_especial', 'data_situacao_especial'
]
colunas_simples = [
    'cnpj_basico', 'opcao_pelo_simples', 'data_opcao_pelo_simples', 'data_exclusao_do_simples',
    'opcao_pelo_mei', 'data_opcao_pelo_mei', 'data_exclusao_do_mei'
]

In [ ]:
def ler_familia(pasta: Path, colunas: list[str]) -> pl.DataFrame:
    arquivos = sorted([arquivo for arquivo in pasta.iterdir() if arquivo.is_file()])
    consultas = []
    for arquivo in arquivos:
        consultas.append(
            pl.scan_csv(
                arquivo,
                separator=';',
                has_header=False,
                quote_char='"',
                new_columns=colunas,
                infer_schema_length=0,
                encoding='latin1',
            )
        )
    return pl.concat(consultas).collect(streaming=True)

empresas = ler_familia(SNAPSHOT / '02_extraido_texto' / 'empresas', colunas_empresas)
estabelecimentos = ler_familia(SNAPSHOT / '02_extraido_texto' / 'estabelecimentos', colunas_estabelecimentos)
simples = ler_familia(SNAPSHOT / '02_extraido_texto' / 'simples', colunas_simples)

empresas.shape, estabelecimentos.shape, simples.shape

In [ ]:
recorte_estabelecimentos = estabelecimentos.filter(pl.col('municipio') == CODIGO_MUNICIPIO)
recorte_empresas = empresas.join(recorte_estabelecimentos.select('cnpj_basico').unique(), on='cnpj_basico', how='inner')
recorte_simples = simples.join(recorte_estabelecimentos.select('cnpj_basico').unique(), on='cnpj_basico', how='inner')

recorte_estabelecimentos.shape, recorte_empresas.shape, recorte_simples.shape

In [ ]:
saida_bronze = SNAPSHOT / '03_processado' / 'bronze'
saida_bronze.mkdir(parents=True, exist_ok=True)

recorte_estabelecimentos.write_parquet(saida_bronze / 'estabelecimentos_municipio.parquet')
recorte_empresas.write_parquet(saida_bronze / 'empresas_municipio.parquet')
recorte_simples.write_parquet(saida_bronze / 'simples_municipio.parquet')

print('Artefatos bronze gerados com sucesso.')